# M3.4 · Optimizer'lar: SGD, momentum, RMSProp, Adam, AdamW

Adam'ın iki adımını bias düzeltmesiyle elle atıyor, saf Python'da dört optimizer'ı PyTorch'un formülleriyle birebir yazıyor, NumPy'da Adam ile AdamW farkını ölçüyor, PyTorch'ta `torch.optim.Optimizer`'dan türeyen kendi AdamW'ını yazıp resmi olanla bit düzeyinde eşleştiriyor ve en sonda `_foreach` çok-tensörlü işlemlerle parametre döngüsünü kaldırıyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m3-4-optimizers.html](https://m-gonul.github.io/pytorch-lab/m3-4-optimizers.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m3_4_optimizers.py
CHECKS = {
 "b": "import math\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef rosen_grad(w):\n    x, y = w\n    return [-2 * (1 - x) - 400 * x * (y - x * x), 200 * (y - x * x)]\n\n\ndef ref_traj(kind, steps, **kw):\n    w, st = [-1.2, 1.0], {}\n    out = []\n    for _ in range(steps):\n        g = rosen_grad(w)\n        if kind == \"sgd\":\n            u = st.get(\"u\"); u = list(g) if u is None else [kw[\"beta\"] * a + b for a, b in zip(u, g)]; st[\"u\"] = u\n            w = [a - kw[\"lr\"] * b for a, b in zip(w, u)]\n        elif kind == \"rms\":\n            s = st.get(\"s\", [0.0, 0.0]); s = [0.99 * a + 0.01 * b * b for a, b in zip(s, g)]; st[\"s\"] = s\n            w = [a - kw[\"lr\"] * b / (math.sqrt(c) + 1e-8) for a, b, c in zip(w, g, s)]\n        else:\n            t = st.get(\"t\", 0) + 1; m = st.get(\"m\", [0.0, 0.0]); v = st.get(\"v\", [0.0, 0.0])\n            if kw.get(\"decoupled\"):\n                w = [a * (1 - kw[\"lr\"] * kw[\"wd\"]) for a in w]\n            else:\n                g = [a + kw[\"wd\"] * b for a, b in zip(g, w)]\n            m = [0.9 * a + 0.1 * b for a, b in zip(m, g)]; v = [0.999 * a + 0.001 * b * b for a, b in zip(v, g)]\n            st.update(t=t, m=m, v=v)\n            w = [a - kw[\"lr\"] * (b / (1 - 0.9 ** t)) / (math.sqrt(c / (1 - 0.999 ** t)) + 1e-8) for a, b, c in zip(w, m, v)]\n        out.append(list(w))\n    return out\n\n\ndef run(fn, steps, **kw):\n    w, st, out = [-1.2, 1.0], {}, []\n    for _ in range(steps):\n        w = fn(w, rosen_grad(w), st, **kw); out.append(list(w))\n    return out\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy/torch kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n\n    def hand():\n        w, st = [0.0], {}\n        ws = []\n        for _ in range(2):\n            w = need(ns, \"adam\")(w, [2 * (w[0] - 3)], st, lr=0.1, eps=0.0); ws.append(w[0])\n        close(ws, [0.1, 0.199897], atol=2e-6)\n    R.test(\"Adam elle örnek: 0.1, 0.1999\", hand)\n    R.test(\"SGD momentum: Rosenbrock yörüngesi\", lambda: close(run(need(ns, \"sgd_momentum\"), 30, lr=1e-4, beta=0.9), ref_traj(\"sgd\", 30, lr=1e-4, beta=0.9)))\n    R.test(\"RMSProp: Rosenbrock yörüngesi\", lambda: close(run(need(ns, \"rmsprop\"), 30, lr=1e-3), ref_traj(\"rms\", 30, lr=1e-3)))\n    R.test(\"Adam (L2, wd = 0.1)\", lambda: close(run(need(ns, \"adam\"), 30, lr=1e-2, wd=0.1), ref_traj(\"adam\", 30, lr=1e-2, wd=0.1)))\n    R.test(\"AdamW (decoupled, wd = 0.1)\", lambda: close(run(need(ns, \"adam\"), 30, lr=1e-2, wd=0.1, decoupled=True), ref_traj(\"adam\", 30, lr=1e-2, wd=0.1, decoupled=True)))\n",
 "c": "import numpy as np\nfrom labcheck import close, max_loops, no_imports, need, ensure\n\n\ndef ref(params, grad_fn, steps, lr, wd, decoupled, b1=0.9, b2=0.999, eps=1e-8):\n    p = {k: v.copy() for k, v in params.items()}\n    m = {k: np.zeros_like(v) for k, v in p.items()}; v_ = {k: np.zeros_like(v) for k, v in p.items()}\n    for t in range(1, steps + 1):\n        g = grad_fn(p)\n        for k in p:\n            gk = g[k] if decoupled else g[k] + wd * p[k]\n            if decoupled:\n                p[k] = p[k] * (1 - lr * wd)\n            m[k] = b1 * m[k] + (1 - b1) * gk; v_[k] = b2 * v_[k] + (1 - b2) * gk * gk\n            p[k] = p[k] - lr * (m[k] / (1 - b1 ** t)) / (np.sqrt(v_[k] / (1 - b2 ** t)) + eps)\n    return p\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"step: en fazla bir (parametre) döngüsü\", lambda: max_loops(src, 1, func=\"step\"))\n    R.test(\"torch kullanılmıyor\", lambda: no_imports(src, {\"torch\"}))\n    rng = np.random.default_rng(0)\n    A, B = rng.normal(size=(20, 5)), rng.normal(size=20)\n    gf = lambda p: {\"W\": 2 * A.T @ (A @ p[\"W\"] + p[\"b\"] - B) / 20, \"b\": np.array([2 * (A @ p[\"W\"] + p[\"b\"] - B).mean()])}\n    init = {\"W\": rng.normal(size=5), \"b\": np.array([0.5])}\n    Adam = need(ns, \"Adam\", \"sınıf\") if \"Adam\" in ns else None\n\n    def run(decoupled):\n        p = {k: v.copy() for k, v in init.items()}\n        opt = ns[\"Adam\"](p, lr=0.05, weight_decay=0.1, decoupled=decoupled)\n        for _ in range(40):\n            opt.step(gf(p))\n        return p\n\n    def traj():\n        need(ns, \"Adam\", \"sınıf\")\n        for dec in (False, True):\n            p = run(dec); want = ref(init, gf, 40, 0.05, 0.1, dec)\n            close(p[\"W\"], want[\"W\"], what=f\"decoupled={dec} W\"); close(p[\"b\"], want[\"b\"], what=f\"decoupled={dec} b\")\n        pa, pw = run(False), run(True)\n        return f\"40 adım sonra ‖W‖: Adam (L2) {np.linalg.norm(pa['W']):.4f}, AdamW {np.linalg.norm(pw['W']):.4f}\"\n    R.test(\"Adam ve AdamW referansla aynı; sonuçları farklı\", traj)\n\n    def ill():\n        need(ns, \"Adam\", \"sınıf\")\n        D = np.array([1.0, 100.0])\n        loss = lambda w: (D * (w - 1) ** 2).sum()\n        w_a = {\"w\": np.zeros(2)}; opt = ns[\"Adam\"](w_a, lr=0.05)\n        w_s = np.zeros(2)\n        for _ in range(200):\n            opt.step({\"w\": 2 * D * (w_a[\"w\"] - 1)})\n            w_s = w_s - 0.009 * 2 * D * (w_s - 1)\n        ensure(loss(w_a[\"w\"]) < 0.1 * loss(w_s), f\"Adam {loss(w_a['w']):.3g}, SGD {loss(w_s):.3g}\")\n        return f\"κ = 100 kuadratik, 200 adım: Adam kaybı {loss(w_a['w']):.2e}, SGD (kararlı en büyük lr) {loss(w_s):.2e}\"\n    R.test(\"kötü koşullu problemde Adam SGD'den hızlı\", ill)\n",
 "d": "import copy\nimport torch\nfrom torch import nn\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    def eq():\n        Cls = need(ns, \"MyAdamW\", \"sınıf\")\n        ensure(issubclass(Cls, torch.optim.Optimizer), \"torch.optim.Optimizer'dan türemeli\")\n        torch.manual_seed(0)\n        m1 = nn.Sequential(nn.Linear(4, 16), nn.Tanh(), nn.Linear(16, 3)).double()\n        m2 = copy.deepcopy(m1)\n        groups = lambda m: [{\"params\": [m[0].weight, m[2].weight], \"weight_decay\": 0.05}, {\"params\": [m[0].bias, m[2].bias], \"weight_decay\": 0.0}]\n        o1 = Cls(groups(m1), lr=0.01, betas=(0.85, 0.99), eps=1e-7)\n        o2 = torch.optim.AdamW(groups(m2), lr=0.01, betas=(0.85, 0.99), eps=1e-7)\n        X = torch.randn(32, 4, dtype=torch.float64); y = torch.randint(0, 3, (32,))\n        for _ in range(30):\n            for m, o in ((m1, o1), (m2, o2)):\n                o.zero_grad()\n                nn.functional.cross_entropy(m(X), y).backward()\n                o.step()\n        for (n, a), (_, b) in zip(m1.named_parameters(), m2.named_parameters()):\n            close(a, b, rtol=1e-12, atol=1e-14, what=n)\n        st = o1.state[m1[0].weight]\n        ensure(int(st[\"step\"]) == 30, \"state['step'] 30 olmalı\")\n        return \"30 adım, 2 param_group: torch.optim.AdamW ile 12 basamak aynı\"\n    R.test(\"MyAdamW = torch.optim.AdamW\", eq)\n\n    def clo():\n        Cls = need(ns, \"MyAdamW\", \"sınıf\")\n        w = torch.zeros(2, requires_grad=True)\n        o = Cls([w], lr=0.1, weight_decay=0.0)\n        def closure():\n            o.zero_grad(); L = ((w - 3) ** 2).sum(); L.backward(); return L\n        L = o.step(closure)\n        ensure(L is not None and abs(float(L.detach()) - 18) < 1e-9, \"step(closure) kaybı döndürmeli\")\n        close(w.detach(), [0.1, 0.1], atol=1e-6, what=\"ilk adım ≈ lr\")\n    R.test(\"step(closure) destekleniyor\", clo)\n",
 "e": "import math\nimport torch\nfrom labcheck import close, max_loops, need, ensure, bench, fmt_time\n\n\n@torch.no_grad()\ndef loop_step(params, grads, ms, vs, t, lr, b1, b2, eps, wd):\n    for p, g, m, v in zip(params, grads, ms, vs):\n        p.mul_(1 - lr * wd); m.mul_(b1).add_(g, alpha=1 - b1); v.mul_(b2).addcmul_(g, g, value=1 - b2)\n        p.addcdiv_(m, v.sqrt() / math.sqrt(1 - b2 ** t) + eps, value=-lr / (1 - b1 ** t))\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"parametreler üzerinde döngü yok\", lambda: max_loops(src, 0, func=\"adamw_foreach_\"))\n    g = torch.Generator().manual_seed(0)\n    shapes = [(3, 4), (4,), (7, 2, 2), (1,)]\n\n    def eq():\n        P1 = [torch.randn(s, generator=g, dtype=torch.float64) for s in shapes]\n        P2 = [p.clone() for p in P1]\n        M1, V1 = [torch.zeros_like(p) for p in P1], [torch.zeros_like(p) for p in P1]\n        M2, V2 = [torch.zeros_like(p) for p in P1], [torch.zeros_like(p) for p in P1]\n        for t in range(1, 11):\n            G = [torch.randn(s, generator=g, dtype=torch.float64) for s in shapes]\n            need(ns, \"adamw_foreach_\")(P1, G, M1, V1, t, 0.01, 0.9, 0.999, 1e-8, 0.05)\n            loop_step(P2, G, M2, V2, t, 0.01, 0.9, 0.999, 1e-8, 0.05)\n        for a, b in zip(P1, P2):\n            close(a, b, rtol=1e-12, atol=1e-14)\n        for a, b in zip(V1, V2):\n            close(a, b, rtol=1e-12, atol=1e-14, what=\"exp_avg_sq\")\n    R.test(\"10 adım: döngülü AdamW ile aynı (yerinde)\", eq)\n    if \"adamw_foreach_\" not in ns:\n        return\n    P = [torch.randn(64) for _ in range(500)]\n    G = [torch.randn(64) for _ in range(500)]\n    M, V = [torch.zeros(64) for _ in P], [torch.zeros(64) for _ in P]\n    t_loop = bench(lambda: loop_step(P, G, M, V, 1, 1e-3, 0.9, 0.999, 1e-8, 0.01), repeat=5)\n    t_fe = bench(lambda: ns[\"adamw_foreach_\"](P, G, M, V, 1, 1e-3, 0.9, 0.999, 1e-8, 0.01), repeat=5)\n    R.table(\"AdamW adımı, 500 tensör × 64 eleman\", [\"yöntem\", \"süre\", \"döngüye göre\"],\n            [[\"for p in params (≈ 3000 kernel çağrısı)\", fmt_time(t_loop), \"1×\"], [\"senin adamw_foreach_\", fmt_time(t_fe), f\"{t_loop / t_fe:,.1f}× hızlı\"]],\n            note=\"GPU'da fark çok daha büyük: foreach işlemleri tensörleri tek bir kernel'da toplar (multi-tensor apply).\")\n",
 "x1": "import numpy as np\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        close(need(ns, \"ema\")(np.array([1.0, 1.0, 1.0]), 0.9), [0.1, 0.19, 0.271], what=\"EMA\")\n        close(ns[\"ema\"](np.array([2.0, 2.0, 2.0]), 0.9, True), [2.0, 2.0, 2.0], what=\"bias düzeltmesi sabit sinyali korur\")\n        rng = np.random.default_rng(0)\n        T = 2000\n        trend = np.sin(np.arange(T) / 150)\n        g = trend + rng.normal(0, 1.0, T)\n        rows = []\n        for b in (0.01, 0.5, 0.9, 0.99):\n            f = ns[\"ema\"](g, b, True)\n            noise = np.std(f[300:] - trend[300:])\n            lag = int(np.argmax([np.corrcoef(f[300 + k:1800 + k], trend[300:1800])[0, 1] for k in range(0, 120)]))\n            rows.append([b, f\"{1 / (1 - b):.0f}\", f\"{noise:.3f}\", lag])\n        ensure(float(rows[0][2]) > 0.9 and float(rows[2][2]) < 0.4, \"β büyüdükçe gürültü azalır\")\n        ensure(rows[3][3] > rows[1][3], \"β → 1: gecikme artar\")\n        R_.table(\"Dalgalı gradyan = yavaş trend + N(0, 1) gürültü (ders alıştırması 3)\", [\"unutma faktörü β\", \"etkin pencere 1/(1 − β)\", \"trend etrafındaki hata std\", \"gecikme (adım)\"], rows,\n                 note=\"β = 0.01'de filtre neredeyse hiçbir şeyi hatırlamaz — sinyal ham gradyan kadar gürültülü. β = 0.99'da son ~100 gradyanın ortalaması: gürültü kaybolur ama trend değişimleri gecikmeli izlenir. 'Unutma faktörü' adı bundan: β, eski gradyanların her adımda ne kadarının hatırlandığıdır.\")\n        A = np.diag([1.0, 100.0])\n        x0 = np.array([1.0, 1.0])\n        k = 100.0\n        lr = 4 / (1 + np.sqrt(k)) ** 2\n        beta = ((np.sqrt(k) - 1) / (np.sqrt(k) + 1)) ** 2\n        hb = need(ns, \"heavy_ball\")(A, x0, lr, beta, 150)\n        ne = need(ns, \"nesterov\")(A, x0, 1 / 100, (np.sqrt(k) - 1) / (np.sqrt(k) + 1), 150)\n        sgd = ns[\"heavy_ball\"](A, x0, 2 / 101, 0.0, 150)\n        ensure(hb[-1] < 1e-8 * hb[0] and sgd[-1] > 1e-3 * sgd[0] and ne[-1] < 1e-4 * ne[0], \"momentum koşul sayısının karekökü kadar hızlandırır\")\n        R_.note(f\"κ = 100'lük kuadratikte 150 adım sonra kayıp: SGD (en iyi η) {sgd[-1]:.2e}, ağır top (Polyak, en iyi η, β) {hb[-1]:.2e}, Nesterov (η = 1/L, β = (√κ − 1)/(√κ + 1)) {ne[-1]:.2e}. Yakınsama oranı SGD'de ≈ 1 − 2/κ, momentumla ≈ 1 − 2/√κ.\")\n    R_.test(\"EMA, ağır top, Nesterov\", t)\n",
 "x2": "import numpy as np\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        st = {}\n        w = need(ns, \"adagrad_step\")(np.array([1.0, 1.0]), np.array([2.0, -0.5]), st, 0.1)\n        close(w, [0.9, 1.1], what=\"AdaGrad ilk adım: işaret × η\")\n        w = ns[\"adagrad_step\"](w, np.array([2.0, -0.5]), st, 0.1)\n        close(w, [0.9 - 0.1 * 2 / np.sqrt(8), 1.1 + 0.1 * 0.5 / np.sqrt(0.5)], what=\"birikmiş G\")\n        st = {}\n        w = need(ns, \"adamax_step\")(np.array([0.0]), np.array([-4.0]), st, 0.1)\n        close(w, [0.1], what=\"AdaMax ilk adım\")\n        w = ns[\"adamax_step\"](w, np.array([1.0]), st, 0.1)\n        m = 0.9 * (-0.4) + 0.1 * 1\n        close(w, [0.1 - 0.1 / (1 - 0.81) * m / (0.999 * 4.0)], atol=1e-7, what=\"u = max(β₂u, |g|)\")\n        st = {}\n        w = need(ns, \"nadam_step\")(np.array([0.0]), np.array([-6.0]), st, 0.1)\n        mh = 0.9 * (-0.6) / (1 - 0.81) + 0.1 * (-6) / 0.1\n        close(w, [-0.1 * mh / 6.0], atol=1e-7, what=\"Nadam ilk adım\")\n        rng = np.random.default_rng(0)\n        N, d = 4000, 50\n        X = (rng.random((N, d)) < 0.5).astype(float)\n        X[:, 0] = rng.random(N) < 0.01\n        wt = np.zeros(d)\n        wt[0], wt[1:] = 6.0, rng.normal(0, 0.3, d - 1)\n        y = (rng.random(N) < 1 / (1 + np.exp(-(X @ wt - 0.5 * (X[:, 1:] @ wt[1:]).mean())))).astype(float)\n        rows = []\n        res = {}\n        for name, step, lr in ((\"SGD\", None, 0.1), (\"AdaGrad\", ns[\"adagrad_step\"], 0.1), (\"AdaMax\", ns[\"adamax_step\"], 0.02), (\"Nadam\", ns[\"nadam_step\"], 0.02)):\n            w, st = np.zeros(d), {}\n            for ep in range(3):\n                for i in range(0, N, 20):\n                    xb, yb = X[i:i + 20], y[i:i + 20]\n                    g = xb.T @ (1 / (1 + np.exp(-xb @ w)) - yb) / len(xb)\n                    w = w - lr * g if step is None else step(w, g, st, lr)\n            res[name] = w[0]\n            rows.append([name, lr, f\"{w[0]:.2f}\", f\"{np.corrcoef(w[1:], wt[1:])[0, 1]:.2f}\"])\n        ensure(res[\"AdaGrad\"] > 2 * res[\"SGD\"], \"seyrek öznitelik: uyarlamalı yöntem nadir özniteliğin ağırlığını hızla öğrenir\")\n        R_.table(\"Lojistik regresyon, öznitelik 0 örneklerin yalnız %1'inde açık (gerçek ağırlık 6), 3 epoch\", [\"yöntem\", \"η\", \"w₀ (nadir öznitelik)\", \"corr(w, w*) diğerleri\"], rows,\n                 note=\"SGD'de nadir öznitelik nadiren gradyan alır ve her seferinde aynı küçük η ile ilerler. AdaGrad her koordinatı geçmiş gradyan büyüklüğüne böler: az gradyan görmüş koordinat büyük adım atar (seyrek veride — kelimeler, tıklamalar — tarihsel başarısı). Bedeli: G hiç küçülmediği için uzun eğitimde öğrenme oranı sıfıra iner; RMSProp/Adam EMA ile bunu düzeltir.\")\n    R_.test(\"AdaGrad, AdaMax, Nadam\", t)\n",
 "x3": "import numpy as np\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        close(need(ns, \"power\")(20, 0.01, 20), 0.005, what=\"power\")\n        close(need(ns, \"exponential\")(40, 0.01, 20), 1e-4, what=\"exponential\")\n        ensure([need(ns, \"piecewise\")(t, [5, 15], [0.01, 0.005, 0.001]) for t in (0, 5, 14, 15, 99)] == [0.01, 0.005, 0.005, 0.001, 0.001], \"piecewise\")\n        oc = [need(ns, \"one_cycle\")(t, 100, 0.01, 0.1) for t in range(101)]\n        close((oc[0], oc[45], oc[90], oc[100]), (0.01, 0.1, 0.01, 1e-5), atol=1e-9, what=\"1cycle kilit noktaları\")\n        ensure(need(ns, \"plateau\")([5, 4, 4, 4, 4, 3, 3, 3, 3], 1.0, 0.5, 3) == [1, 1, 1, 1, 0.5, 0.5, 0.5, 0.5, 0.25], \"plateau\")\n        rng = np.random.default_rng(0)\n        lam = np.logspace(0, 2, 20)\n        T = 300\n        rows = []\n        res = {}\n        for name, sched in ((\"sabit 0.015\", lambda t: 0.015), (\"power (s = 30)\", lambda t: ns[\"power\"](t, 0.015, 30)), (\"üstel (s = 150)\", lambda t: ns[\"exponential\"](t, 0.015, 150)),\n                            (\"piecewise\", lambda t: ns[\"piecewise\"](t, [150, 250], [0.015, 0.003, 0.0006])), (\"1cycle\", lambda t: ns[\"one_cycle\"](t, T, 0.0015, 0.019))):\n            x = np.ones(20)\n            for t_ in range(T):\n                g = lam * x + rng.normal(0, 3, 20)\n                x = x - sched(t_) * g\n            res[name] = 0.5 * np.sum(lam * x ** 2)\n            rows.append([name, f\"{res[name]:.3f}\"])\n        ensure(res[\"1cycle\"] < res[\"sabit 0.015\"] and res[\"piecewise\"] < res[\"sabit 0.015\"], \"azalan oran gürültü tabanını düşürür\")\n        R_.table(\"Gürültülü kuadratik (κ = 100, gradyan gürültüsü σ = 3), 300 SGD adımı: son kayıp\", [\"takvim\", \"kayıp\"], rows,\n                 note=\"Sabit oran gürültü tabanında takılır (kayıp ∝ η·σ²). Azalan takvimler önce hızlı ilerleyip sonra gürültüyü ortalar. 1cycle (Smith) ortada büyük oranla 'keşfeder', sonunda oranı birkaç mertebe düşürerek ince ayar yapar — Géron'un önerisi: aynı epoch sayısında çoğu zaman en iyisi.\")\n    R_.test(\"öğrenme oranı takvimleri\", t)\n"
}

In [ ]:
import importlib, labcheck, checks_m3_4_optimizers as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

Hepsi aynı iskelet: gradyandan bir **durum** güncelle, durumdan bir adım hesapla. PyTorch'un formülleriyle ($g_t$ adım $t$'deki gradyan):

| yöntem | durum | adım |
|---|---|---|
| SGD + momentum | $u\leftarrow\beta u+g$ | $w\leftarrow w-\eta u$ |
| RMSProp | $s\leftarrow\alpha s+(1-\alpha)g^2$ | $w\leftarrow w-\eta\,g/(\sqrt s+\epsilon)$ |
| Adam | $m\leftarrow\beta_1m+(1-\beta_1)g$, $v\leftarrow\beta_2v+(1-\beta_2)g^2$ | $w\leftarrow w-\eta\,\frac{\hat m}{\sqrt{\hat v}+\epsilon}$ |
| AdamW | Adam ile aynı | önce $w\leftarrow w(1-\eta\lambda)$, sonra Adam |

Adam'ın bias düzeltmesi: $\hat m=m/(1-\beta_1^t)$, $\hat v=v/(1-\beta_2^t)$. Sıfırdan başlayan hareketli ortalamaların ilk adımlardaki küçüklüğünü telafi eder.

RMSProp ve Adam her parametreyi kendi gradyan geçmişinin büyüklüğüyle ölçekler: sık ve büyük gradyan alan parametreler küçük, nadir gradyan alanlar büyük adımlar atar. Adam'da (L2) `weight_decay` gradyana $\lambda w$ ekler; AdamW ise decay'i ölçekleme dışında, doğrudan parametreye uygular.

## 1 · Elle

<!-- çözüm -->

İlk Adam adımı ilginç: bias düzeltmesinden sonra $\hat m_1/\sqrt{\hat v_1}=g_1/|g_1|=-1$, yani ilk adım gradyanın büyüklüğünden bağımsız olarak tam $\eta$. Adam'ın adım büyüklüğü yaklaşık $\eta$ ile sınırlıdır; bu, lr seçimini kolaylaştırır.

İkinci adımda da $\approx\eta$: gradyan yönü değişmediği sürece Adam sabit hızla ilerler. Düz GD'nin aksine minimuma yaklaşırken (gradyan küçülürken) kendiliğinden yavaşlamaz; bu yüzden pratikte lr schedule'ı ile birlikte kullanılır.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**Adam, $L(w)=(w-3)^2$, $w_0=0$, $\eta=0.1$, $\beta_1=0.9$, $\beta_2=0.999$, $\epsilon=0$. Adım 1: $g_1=-6$**

- `m1`: $m_1=(1-\beta_1)g_1$
- `v1`: $v_1=(1-\beta_2)g_1^2$
- `mh1`: $\hat m_1=m_1/(1-\beta_1^1)$
- `w1`: $w_1=w_0-\eta\,\hat m_1/\sqrt{\hat v_1}$

**Adım 2: $g_2=2(w_1-3)$**

- `m2`: $m_2$
- `w2`: $w_2$

**AdamW ayrık weight decay, $\lambda=0.01$, $\eta=0.1$**

- `wd`: $w=2$ için yalnız decay adımı $w(1-\eta\lambda)$

In [ ]:
cevaplar = dict(
    m1=None,
    v1=None,
    mh1=None,
    w1=None,
    m2=None,
    w2=None,
    wd=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"m1":[-0.6,0.0005],"v1":[0.036,0.0001],"mh1":[-6,0.001],"w1":[0.1,0.0005],"m2":[-1.12,0.001],"w2":[0.199897,0.0005],"wd":[1.998,0.0001]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python; parametre ve gradyan **liste**, durum bir **sözlük** (ilk çağrıda boş gelir, fonksiyon kendisi doldurur). Her fonksiyon yeni parametre listesini döndürür.

- `sgd_momentum(w, g, state, lr, beta=0.9)`
- `rmsprop(w, g, state, lr=0.01, alpha=0.99, eps=1e-8)`
- `adam(w, g, state, lr=1e-3, b1=0.9, b2=0.999, eps=1e-8, wd=0.0, decoupled=False)` — `decoupled=False`: `g += wd·w` (Adam); `True`: `w *= 1 - lr·wd` (AdamW). Adım sayacı `state["t"]`.

Kontrol, yörüngeleri PyTorch formüllerinden üretilmiş referansla karşılaştırır.

In [ ]:
%%writefile stage_b.py
import math


def sgd_momentum(w, g, state, lr, beta=0.9):
    ...


def rmsprop(w, g, state, lr=0.01, alpha=0.99, eps=1e-8):
    ...


def adam(w, g, state, lr=1e-3, b1=0.9, b2=0.999, eps=1e-8, wd=0.0, decoupled=False):
    ...


w, st = [0.0], {}
for _ in range(2):
    w = adam(w, [2 * (w[0] - 3)], st, lr=0.1, eps=0.0)
print(w)       # [0.19989…]

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy'da sözlük tabanlı bir Adam: parametreler `{"W": array, …}`.

- `class Adam` — `__init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.0, decoupled=False)`; `step(self, grads)` parametreleri **yerinde** günceller. Parametre sözlüğü üzerinde tek döngü serbest; eleman döngüsü yok.

Kontrol: (1) referansla birebir yörünge, (2) aynı $\lambda$ ile Adam (L2) ve AdamW'ın farklı ağırlık normlarına gittiği bir problem, (3) kötü koşullu bir kuadratikte Adam'ın aynı adım sayısında SGD'den çok daha düşük kayba indiği.

In [ ]:
%%writefile stage_c.py
import numpy as np


class Adam:
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.0, decoupled=False):
        ...

    def step(self, grads):
        ...


p = {"w": np.zeros(2)}
opt = Adam(p, lr=0.1)
for _ in range(100):
    opt.step({"w": 2 * (p["w"] - np.array([3.0, -1.0]))})
print(p["w"])

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

PyTorch optimizer'ları ortak bir taban sınıftan türer:

```python
class MyAdamW(torch.optim.Optimizer):
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=1e-2):
        super().__init__(params, dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay))

    @torch.no_grad()
    def step(self, closure=None):
        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is None: continue
                state = self.state[p]          # ilk sefer boş sözlük
                ...
```

- `MyAdamW` — `torch.optim.AdamW` ile **aynı** sayıları vermeli (float64, 30 adım, bir MLP'nin tüm parametreleri, iki farklı `param_group`). PyTorch'un formülü: `denom = (v.sqrt() / sqrt(bc2)) + eps`, `p -= lr/bc1 · m / denom`.

In [ ]:
%%writefile stage_d.py
import math
import torch


class MyAdamW(torch.optim.Optimizer):
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=1e-2):
        super().__init__(params, dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay))

    @torch.no_grad()
    def step(self, closure=None):
        ...

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

Yüzlerce parametre tensörlü bir modelde optimizer adımı bir Python döngüsüdür; her tensör için 6–8 ayrı kernel çağrısı. `torch._foreach_*` işlemleri bir **tensör listesini** tek çağrıda işler:

```python
torch._foreach_mul_(exp_avgs, b1)
torch._foreach_add_(exp_avgs, grads, alpha=1 - b1)
torch._foreach_addcmul_(exp_avg_sqs, grads, grads, value=1 - b2)
denom = torch._foreach_sqrt(exp_avg_sqs)
torch._foreach_addcdiv_(params, exp_avgs, denom, value=-step_size)
```

- `adamw_foreach_(params, grads, exp_avgs, exp_avg_sqs, step, lr, b1, b2, eps, wd)` — tüm listeyi **yerinde** günceller (AdamW). **Parametreler üzerinde döngü yok.** `step` 1'den başlayan adım numarası.

Kontrol, döngülü referansla karşılaştırır ve 500 küçük tensörde hız tablosu üretir.

In [ ]:
%%writefile stage_e.py
import math
import torch


@torch.no_grad()
def adamw_foreach_(params, grads, exp_avgs, exp_avg_sqs, step, lr, b1, b2, eps, wd):
    """Tüm listeleri yerinde güncelle. Parametreler üzerinde döngü yok."""
    ...

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · Unutma faktörü, ağır top ve Nesterov momentumu (ders alıştırması 3)

**Ders alıştırması 3, kısım 4:** dalgalı bir gradyan sinyalini momentum ile filtrele, "unutma faktörü" β'yı 0.01 ile 0.99 arasında değiştir. $m_t=\beta m_{t-1}+(1-\beta)g_t$ üstel hareketli ortalamadır (EMA): geçmiş gradyanların ağırlığı $\beta^k$ ile söner — β küçükse çabuk unutur (filtre zayıf), β → 1'de uzun hafıza (yumuşak ama gecikmeli). Etkin pencere ≈ $1/(1-\beta)$.

NumPy.

- `ema(g, beta, bias_correct=False)` — `g (T,)`; `bias_correct=True` ise $m_t/(1-\beta^t)$ (Adam'daki düzeltme). Zaman döngüsü serbest.
- `heavy_ball(A, x0, lr, beta, steps)` ve `nesterov(A, x0, lr, beta, steps)` — $f(x)=\tfrac12x^\top Ax$ üzerinde; Nesterov gradyanı ileri bakılan noktada alır: $v\leftarrow\beta v-\eta\nabla f(x+\beta v)$, $x\leftarrow x+v$. Kayıp geçmişini döndür.

In [ ]:
%%writefile extra_x1.py
import numpy as np


def ema(g, beta, bias_correct=False):
    ...


def heavy_ball(A, x0, lr, beta, steps):
    ...


def nesterov(A, x0, lr, beta, steps):
    ...

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · AdaGrad, AdaMax, Nadam ve seyrek öznitelikler

Uyarlamalı yöntemlerin ailesi (Géron Bölüm 11). NumPy, durum sözlüğü ilk çağrıda boş.

- `adagrad_step(w, g, st, lr=0.1, eps=1e-10)` — $G\leftarrow G+g^2$, $w\leftarrow w-\eta g/(\sqrt G+\epsilon)$.
- `adamax_step(w, g, st, lr=2e-3, b1=0.9, b2=0.999, eps=1e-8)` — $u\leftarrow\max(\beta_2u,|g|)$, $w\leftarrow w-\frac{\eta}{1-\beta_1^t}\frac{m}{u+\epsilon}$ (ℓ∞ normlu Adam).
- `nadam_step(w, g, st, lr=2e-3, b1=0.9, b2=0.999, eps=1e-8)` — Adam + Nesterov (Dozat 2016, sabit β₁): $\hat m=\beta_1\frac{m_t}{1-\beta_1^{t+1}}+(1-\beta_1)\frac{g_t}{1-\beta_1^t}$, $w\leftarrow w-\eta\hat m/(\sqrt{\hat v}+\epsilon)$.

In [ ]:
%%writefile extra_x2.py
import numpy as np


def adagrad_step(w, g, st, lr=0.1, eps=1e-10):
    ...


def adamax_step(w, g, st, lr=2e-3, b1=0.9, b2=0.999, eps=1e-8):
    ...


def nadam_step(w, g, st, lr=2e-3, b1=0.9, b2=0.999, eps=1e-8):
    ...

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

### E3 · Öğrenme oranı takvimleri: power, üstel, parçalı, performans, 1cycle

Öğrenme oranı takvimleri (Géron): t adım sayısı.

- `power(t, lr0, s, c=1)` — $\eta_0/(1+t/s)^c$.
- `exponential(t, lr0, s)` — $\eta_0\cdot0.1^{t/s}$.
- `piecewise(t, bounds, values)` — `bounds` artan eşikler, `values` bir fazla eleman.
- `one_cycle(t, T, lr0, lr1, tail=0.1)` — ilk $(1-\text{tail})T/2$ adımda $\eta_0\to\eta_1$ doğrusal artış, sonraki eşit sürede $\eta_1\to\eta_0$ doğrusal azalış, son tail·T adımda $\eta_0\to\eta_0/1000$ doğrusal.
- `plateau(losses, lr0, factor=0.5, patience=3)` — performans takvimi: en iyi kayıp `patience` ardışık değerlendirmede iyileşmezse oranı `factor` ile çarp (sayaç sıfırlanır); her değerlendirme için oran listesi.

In [ ]:
%%writefile extra_x3.py
import math


def power(t, lr0, s, c=1):
    ...


def exponential(t, lr0, s):
    ...


def piecewise(t, bounds, values):
    ...


def one_cycle(t, T, lr0, lr1, tail=0.1):
    ...


def plateau(losses, lr0, factor=0.5, patience=3):
    ...

In [ ]:
# Kontrol (E3)
labcheck.run_file("extra_x3.py", CHECKS["x3"])

## Makale ödevi: Sutskever, Martens, Dahl, Hinton (2013): başlatma ve momentumun önemi

[Sutskever, Martens, Dahl, Hinton — On the importance of initialization and momentum in deep learning (ICML 2013)](https://proceedings.mlr.press/v28/sutskever13.html)

Makalenin derin otomatik kodlayıcı deneyini küçük ölçekte tekrarla (MNIST, 784-400-200-100-50-25-6 simetrik AE, sigmoid): seyrek başlatma (her birime 15 rastgele bağlantı) ile klasik momentum ve Nesterov'u, makaledeki μ takvimiyle (Denk. 5: $\mu_t=\min(1-2^{-1-\log_2(\lfloor t/250\rfloor+1)},\mu_{max})$) karşılaştır; μ_max ∈ {0.9, 0.99, 0.995} için Tablo 1'deki gibi eğitim hatasını raporla. Nesterov'un yüksek μ'da daha kararlı olduğunu x1'deki kuadratik analizle ilişkilendir.


**Adımlar**

1. Son 'ince ayar' aşamasında μ'yu 0.9'a düşürmenin etkisini de ölç.

**Kabul ölçütleri**

- Nesterov, yüksek μ_max'ta klasik momentumdan belirgin düşük hata.
- Rastgele (yoğun) başlatma ile seyrek başlatma farkı tabloda.

## Makale ödevi: Smith & Topin (2019): süper yakınsama ve 1cycle

[Smith & Topin — Super-Convergence: Very Fast Training of Neural Networks Using Large Learning Rates (2019)](https://arxiv.org/abs/1708.07120)

Öğrenme oranı aralık testini (LR range test: oranı üstel artırırken kaybı izle) uygula, bundan η₁'i seç; CIFAR-10'da küçük bir ResNet'i aynı epoch bütçesinde sabit oran, adım takvimi ve 1cycle (oran + ters momentum döngüsü 0.95 → 0.85 → 0.95) ile eğit. Makale Şekil 1'deki gibi doğruluk eğrilerini ve son doğrulukları raporla; az veri (CIFAR'ın %10'u) rejiminde farkın büyüdüğünü kontrol et.


**Adımlar**

1. PyTorch'ta OneCycleLR ile kendi x3 uygulamanı karşılaştır.

**Kabul ölçütleri**

- 1cycle, aynı epoch'ta adım takviminden yüksek ya da eşit doğruluk.
- LR aralık testi grafiği ve seçilen η₁ gerekçesi.

## Kaynaklar

- [Kingma & Ba — Adam: A Method for Stochastic Optimization](https://arxiv.org/abs/1412.6980) — arXiv
- [Loshchilov & Hutter — Decoupled Weight Decay Regularization (AdamW)](https://arxiv.org/abs/1711.05101) — arXiv
- [Dive into Deep Learning §12.8–12.10 RMSProp, Adadelta, Adam](https://d2l.ai/chapter_optimization/adam.html) — CC BY-SA 4.0 / MIT-0
- [PyTorch — torch.optim.Adam / AdamW (algoritma sözde kodları)](https://docs.pytorch.org/docs/stable/generated/torch.optim.AdamW.html) — BSD-3